# Localisation of solvers

## Localisation

ExaHyPE couples volumetrically. That does not mean that all solvers have to be
solved everywhere. Indeed, the idea to offer support for multiple solvers arose
originally from the a posteriori limiting of Dumbser et al where a Finite Volume
solver complements ADER-DG solutions where shocks arise.

Most projects that follow these rationale realise the same pattern:
We define two different solvers, and we couple them
with each other. The coupling (and solve) however are local operations:

-   The low-order (typically Finite Volume - we therefore always refer
    to Finite Volumes or FV from hereon) solver is our fallback strategy and we therefore
    neither solve with Finite Volumes everywhere in the computational
    domain nor should do we store the corresponding data.

-   The higher-order scheme is used everywhere where the FV is not
    applied or labelled as valid.

-   The FV and the higher-order solver are not coupled everywhere but only in
    boundary regions.

We will effectively end up with a domain decomposition where parts of
the computational domain are handled by the higher-order solver and the other
parts are handled with FV.
To realise this, we need a marker or oracle, i.e. a magic function
per cell which tells us, similar to a multi-phase flow, which
solver rules in this particular cell.
The
marker will guide us which solver is "valid" in which cell, where one solver
overrules the other one, and where we do not have to store one solver's data
or the other.

The image below illustrates the core idea behind typically coupling approaches
in ExaHyPE. The sizes of the overlap might differ, but the principle is always
the same. The illustration shows the minimal overlap that we can typically
use. Both the upper and the lower mesh illustrate the same mesh. I use colours
to show which solver exists in which mesh cell.

![Overlap of two localised solvers](illustrations/44_localisation.png)

- Solver A is running within the filled cells visualised on the the top. In
  empty cells, A neither solves anything nor does it hold data.
- Solver B is running within the filled cells visualised on the bottom. In
  empty cells, B neither solves anything nor does it hold data.
- Consequently, dark red cells are exclusively solved by A, whereas dark
  green cells are exclusively B.
- In orange cells, A solves its PDE. Afterwards, A's updated solution is
  projected onto B's solution. That is, the blue cells hold B's data, but
  they are not updated by B but instead hold projections of A's data. They
  kind of serve as boundary layer for B.
- In the yellow cells, A does not solve its PDE but holds A's data. After
  each step, the cell's solution by B (light green) is written into A's data.
  The yellow cells serve as Dirichlet boundary data for A's solve.

Starting from this description, sophisticated coupling equations can be
realised, where you exploit the fact that we have a volumetric overlap.

### Localisation guided by a marker

Skip this section if you prefer a hard-coded marking of active regions.

### Avoid that solver data is held everywhere

Once you know where a solver is valid, it makes sense also to only store
the solver's data in the valid region. Otherwise, a lot of memory is "wasted"
which furthermore will be moved/accessed in each and every grid sweep, as the
mesh management of Peano so far does not know that ExaHyPE might not need the
data. ExaHyPE has to let Peano know.

The technical principles behind such a localisation @ref page_peano_localisation "are discussed on a generic Peano page of its own".
ExaHyPE builds up pretty deep inheritance hierarchy
where every subclass specialises and augments the basic numerical scheme, and
also implements additional guards when data is stored or not.
However, most of these subclasses rely on only six routines to control
the data flow:

- _provide_cell_data_to_compute_kernels_default_guard(),
- _load_cell_data_default_guard(),
- _store_cell_data_default_guard(),
- _provide_face_data_to_compute_kernels_default_guard(),
- _load_face_data_default_guard(),
- _store_face_data_default_guard(),

The implementation then uses these predicates to puzzle the actual storage
scheme together via calls similar to

```python
        self._patch.generator.load_store_compute_flag = "::peano4::grid::constructLoadStoreComputeFlag({},{},{})".format(
            self._provide_cell_data_to_compute_kernels_default_guard(),
            self._load_cell_data_default_guard(),
            self._store_cell_data_default_guard(),
        )
```

By default, most solvers only store data for unrefined
cells. They mask coarser levels with in the tree out.
Subclasses such as enclave tasking variations augment the guards
further such that not all data are stored all the time. Nothing stops you
from adding further constraints and to use a solver only in a certain
subdomain for example.